# Verificación de BSPs Absolutas y de Perspectiva

Este notebook verifica que la función `identificador` en `sae/bsp_identifier.py` genere correctamente:
1.  **BSPs Absolutas (B/W)**: Deben ser invariantes independientemente del jugador (`color_jugador`) que las observe.
2.  **BSPs de Perspectiva (1/2)**: Deben cambiar dependiendo del jugador (Mío vs Oponente).

Configuración: Tablero inicial de Othello.

In [ ]:
import sys
import os
import numpy as np

# Configurar path para incluir directorio raíz
current_dir = os.getcwd()
root_dir = os.path.abspath(os.path.join(current_dir, '../..'))
if root_dir not in sys.path:
    sys.path.append(root_dir)

from sae.tools.bsp_identifier import identificador

print(f"Directorio raíz añadido: {root_dir}")

Directorio raíz añadido: c:\Users\karin\Documents\Repos\TIC\othello_world


### Funciones auxiliares
Estas funciones permiten visualizar el tablero y las BSPs activas de manera formateada.

In [19]:
# Funciones auxiliares de visualización (tomadas de board_state_properties.ipynb)

def imprimir_tablero(tablero):
    print("  A B C D E F G H")
    for i in range(8):
        print(f"{i+1}", end=" ")
        for j in range(8):
            valor = tablero[i, j]
            if valor == 0:
                print(".", end=" ")
            elif valor == 1:
                print("●", end=" ")  # Negra
            else:
                print("○", end=" ")  # Blanca
        print(f"{i+1}")
    print("  A B C D E F G H")

def obtener_bsps_activas(bsps):
    """Retorna solo las BSPs que son True"""
    return {k: v for k, v in bsps.items() if v}

def contar_bsps_por_tipo(bsps):
    """Cuenta cuántas BSPs de cada tipo están activas"""
    vacias = sum(1 for k, v in bsps.items() if v and k.endswith('0') and 'BSP_' not in k)
    mias = sum(1 for k, v in bsps.items() if v and k.endswith('1'))
    oponente = sum(1 for k, v in bsps.items() if v and k.endswith('2'))
    absolutas_b = sum(1 for k, v in bsps.items() if v and k.endswith('B'))
    absolutas_w = sum(1 for k, v in bsps.items() if v and k.endswith('W'))
    
    return {
        "vacias": vacias,
        "mias": mias,
        "oponente": oponente,
        "absolutas_b": absolutas_b,
        "absolutas_w": absolutas_w
    }


def imprimir_bsps_activas_con_fichas(bsps, titulo="BSPs ACTIVAS"):
    """Imprime las BSPs activas organizadas en tres apartados: Perspectiva, Absolutas y Estrategia"""
    print("\n" + "=" * 70)
    print(titulo)
    print("=" * 70)
    
    # 1. Separar por tipo
    casillas_mias = []
    casillas_oponente = []
    casillas_absolutas_b = []
    casillas_absolutas_w = []
    propiedades_estrategia = []
    
    for bsp_id, valor in bsps.items():
        if valor:
            if 'BSP_' in bsp_id:
                propiedades_estrategia.append(bsp_id)
            elif bsp_id.endswith('1'):
                casillas_mias.append(bsp_id)
            elif bsp_id.endswith('2'):
                casillas_oponente.append(bsp_id)
            elif bsp_id.endswith('B'):
                casillas_absolutas_b.append(bsp_id)
            elif bsp_id.endswith('W'):
                casillas_absolutas_w.append(bsp_id)
    
    # 2. Imprimir Apartado 1: Perspectiva del Jugador
    print("\n[1] PERSPECTIVA DEL JUGADOR")
    print("-" * 30)
    print(f"Mías ({len(casillas_mias)}):")
    for bsp_id in sorted(casillas_mias):
        casilla = bsp_id[3:-1]
        print(f"  {bsp_id}: Ficha mía en {casilla}")
    
    print(f"\nOponente ({len(casillas_oponente)}):")
    for bsp_id in sorted(casillas_oponente):
        casilla = bsp_id[3:-1]
        print(f"  {bsp_id}: Ficha del oponente en {casilla}")

    # 3. Imprimir Apartado 2: Absolutas
    print("\n\n[2] ABSOLUTAS")
    print("-" * 30)
    print(f"Negras (B) ({len(casillas_absolutas_b)}):")
    for bsp_id in sorted(casillas_absolutas_b):
        casilla = bsp_id[3:-1]
        print(f"  {bsp_id}: Ficha Negra en {casilla}")

    print(f"\nBlancas (W) ({len(casillas_absolutas_w)}):")
    for bsp_id in sorted(casillas_absolutas_w):
        casilla = bsp_id[3:-1]
        print(f"  {bsp_id}: Ficha Blanca en {casilla}")
    
    # 4. Imprimir Apartado 3: Estrategia
    print("\n\n[3] ESTRATEGIA")
    print("-" * 30)
    if propiedades_estrategia:
        for bsp_id in sorted(propiedades_estrategia):
            print(f"  {bsp_id}: True")
    else:
        print("  (Ninguna activa)")
    
    print("=" * 70)

def imprimir_resumen(bsps, titulo="RESUMEN DE BSPs"):
    """Imprime un resumen de las BSPs"""
    print("\n" + titulo)
    print("-" * len(titulo))
    
    conteo = contar_bsps_por_tipo(bsps)
    print(f"Mías: {conteo['mias']} | Oponente: {conteo['oponente']} | Vacías: {conteo['vacias']}")
    print(f"Absolutas B: {conteo['absolutas_b']} | Absolutas W: {conteo['absolutas_w']}")


### Configuración del Tablero
Se crea el tablero inicial de Othello según el estándar.

In [20]:
# Tablero Inicial de Othello
# 0 = Vacío
# 1 = Negro
# -1 = Blanco

tablero = np.zeros((8, 8), dtype=int)

# Configuración Inicial Estándar
# e4 (3,4) = Negro
# d5 (4,3) = Negro
# d4 (3,3) = Blanco
# e5 (4,4) = Blanco

tablero[3, 4] = 1   # e4 Negro
tablero[4, 3] = 1   # d5 Negro
tablero[3, 3] = -1  # d4 Blanco
tablero[4, 4] = -1  # e5 Blanco

print("TABLERO INICIAL:")
imprimir_tablero(tablero)
# Comprobación visual simple
print(f"Negras (1): {np.sum(tablero == 1)}")
print(f"Blancas (-1): {np.sum(tablero == -1)}")

TABLERO INICIAL:
  A B C D E F G H
1 . . . . . . . . 1
2 . . . . . . . . 2
3 . . . . . . . . 3
4 . . . ○ ● . . . 4
5 . . . ● ○ . . . 5
6 . . . . . . . . 6
7 . . . . . . . . 7
8 . . . . . . . . 8
  A B C D E F G H
Negras (1): 2
Blancas (-1): 2


### Ejecución y Visualización
Generamos las BSPs para ambas perspectivas (negro y blanco) y visualizamos los resultados usando las funciones auxiliares.

In [21]:
# Obtener BSPs para ambas perspectivas
bsps_negro = identificador(tablero, 1)    # Juego como Negro
bsps_blanco = identificador(tablero, -1)  # Juego como Blanco

print("BSPs generadas para ambos jugadores.")

# Mostrar Resumen usando helper functions
imprimir_resumen(bsps_negro, "RESUMEN DE BSPs (Perspectiva Negro)")
imprimir_bsps_activas_con_fichas(bsps_negro, "BSPs ACTIVAS (Perspectiva Negro)")

imprimir_resumen(bsps_blanco, "RESUMEN DE BSPs (Perspectiva Blanco)")
imprimir_bsps_activas_con_fichas(bsps_blanco, "BSPs ACTIVAS (Perspectiva Blanco)")

# 1. Comparación de BSPs Absolutas (B o W)
# Deben ser idénticas: Una ficha negra (1) en D4 siempre es 'BSPD4B' independientemente de quién mire.

diffs_absolute = []
abs_keys = [k for k in bsps_negro.keys() if k.endswith('B') or k.endswith('W')]

for k in abs_keys:
    if bsps_negro[k] != bsps_blanco[k]:
        diffs_absolute.append((k, bsps_negro[k], bsps_blanco[k]))

if not diffs_absolute:
    print("\n VERIFICACIÓN DE BSPs ABSOLUTAS: EXITOSA")
    print("Las BSPs terminadas en 'B' (Negro) y 'W' (Blanco) son idénticas para ambos jugadores.")
else:
    print(f"\n ERROR: Se encontraron {len(diffs_absolute)} diferencias en BSPs absolutas.")
    print(diffs_absolute[:10]) # Mostrar primeras 10 diferencias

BSPs generadas para ambos jugadores.

RESUMEN DE BSPs (Perspectiva Negro)
-----------------------------------
Mías: 2 | Oponente: 2 | Vacías: 60
Absolutas B: 2 | Absolutas W: 2

BSPs ACTIVAS (Perspectiva Negro)

[1] PERSPECTIVA DEL JUGADOR
------------------------------
Mías (2):
  BSPD51: Ficha mía en D5
  BSPE41: Ficha mía en E4

Oponente (2):
  BSPD42: Ficha del oponente en D4
  BSPE52: Ficha del oponente en E5


[2] ABSOLUTAS
------------------------------
Negras (B) (2):
  BSPD5B: Ficha Negra en D5
  BSPE4B: Ficha Negra en E4

Blancas (W) (2):
  BSPD4W: Ficha Blanca en D4
  BSPE5W: Ficha Blanca en E5


[3] ESTRATEGIA
------------------------------
  BSP_EN_CURSO: True
  BSP_INICIAL: True

RESUMEN DE BSPs (Perspectiva Blanco)
------------------------------------
Mías: 2 | Oponente: 2 | Vacías: 60
Absolutas B: 2 | Absolutas W: 2

BSPs ACTIVAS (Perspectiva Blanco)

[1] PERSPECTIVA DEL JUGADOR
------------------------------
Mías (2):
  BSPD41: Ficha mía en D4
  BSPE51: Ficha mía en E5